# ⚡ AegisX — Train on Colab (free T4)

Trains AegisX-Mini from scratch and packages it for **manual** upload to
Hugging Face. No auto-push, no token needed in this notebook.

**Your laptop never trains anything.** Run this notebook in Colab:
`File > Upload notebook`, then `Runtime > Run all`.

## 1. Setup

In [ ]:
!pip install -q torch

import os
import sys
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Optional: mount Google Drive so checkpoints survive session disconnects.
# Set USE_DRIVE = False to skip the Google login and save locally instead.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Skipping Drive mount - checkpoints will be lost if the session disconnects.')

## 2. Get the AegisX code

Clone the repo (or upload your local copy with `aegisx/`, `data/`, `targets/`).

In [ ]:
WORK = '/content/aegisx'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

# Clone the repo; pull updates if already cloned (safe to re-run)
if not os.path.isdir('.git'):
    !git clone https://github.com/FerzDevZ/AegisX.git .
else:
    !git -C . pull --ff-only
print('Repo ready:')
print(os.listdir(WORK))

## 2b. Optional: download MORE training data (recommended)

Pulls large public cybersecurity corpora (OWASP cheat sheets, ASVS, MITRE
ATT&CK) into `data/raw/`. More good text = a smarter model.
Each file downloads individually, so a single failure does not stop the rest.

In [ ]:
# Set to True to download public corpora (internet required).
# --cve-pages: NVD recent-CVE pages to fetch (5 pages x 2000 = ~10k CVEs; 0 = skip).
DOWNLOAD_MORE_DATA = True
if DOWNLOAD_MORE_DATA:
    !python scripts/fetch_corpus.py --target-dir data/raw --cve-pages 5
else:
    print('Skipped corpus download.')

# Corpus size report
total = sum(os.path.getsize(os.path.join('data/raw', f)) for f in os.listdir('data/raw') if f.endswith('.txt'))
print(f'Corpus now: {total/1024:.0f} KB of text')

## 3. Configure training

Default: **from scratch** on the corpus in `data/raw/`.

Tuned for the **~46 MB corpus** (fetch §2b) — ~12-14M tokens — and the
upgraded architecture: vocab **8192** (better bilingual EN/ID BPE) and context
**768** tokens (3x longer answers). Lines below: 1 step = 16x4x768 = 49K tokens,
so ~1500 steps ≈ 5 epochs; early stop protects against overfit.

If a checkpoint from an older architecture (vocab/block berbeda) is found in
Drive, this notebook **archives it automatically** and trains fresh with the
new architecture — no crash, no data loss.

In [ ]:
# --- training hyperparameters ---
DATA_DIR      = 'data/raw'
OUT_DIR       = '/content/drive/MyDrive/aegisx/checkpoints/aegisx-mini' if USE_DRIVE else '/content/aegisx/checkpoints/aegisx-mini'
VOCAB_SIZE    = 8192
BLOCK_SIZE    = 768
N_LAYER       = 8
N_HEAD        = 8
N_EMBD        = 512
BATCH_SIZE    = 16
GRAD_ACCUM    = 4
MAX_STEPS     = 1500   # 1 step = 49K token (block 768) ~ 5 epoch korpus 46 MB
LR            = 3e-4
WARMUP_STEPS  = 400
EVAL_EVERY    = 300
SAVE_EVERY    = 100    # simpan model_latest.pt tiap 100 step (tahan putus sesi)
EARLY_STOP    = 5      # stop after 5 evals without val improvement
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- auto-resume: kalau sesi Colab mati, lanjut dari checkpoint berkala ---
RESUME_FROM = ''
for cand in [f'{OUT_DIR}/model_latest.pt', f'{OUT_DIR}/model.pt']:
    if os.path.exists(cand):
        RESUME_FROM = cand
        print('⏳ checkpoint ditemukan - akan RESUME dari', cand)
        break

# --- arsitektur baru (vocab/block) => arsipkan checkpoint lama, mulai dari NOL ---
CFG_JSON = f'{OUT_DIR}/config.json'
if RESUME_FROM and os.path.exists(CFG_JSON):
    try:
        import json as _json
        _old = _json.load(open(CFG_JSON))
        _old_arch = (_old.get('vocab_size'), _old.get('block_size'))
        _new_arch = (VOCAB_SIZE, BLOCK_SIZE)
        if _old_arch != _new_arch:
            import shutil as _sh
            _arc = f"{OUT_DIR}/archive-vocab{_old_arch[0]}-block{_old_arch[1]}"
            os.makedirs(_arc, exist_ok=True)
            for _f in os.listdir(OUT_DIR):
                if _f.startswith(('model', 'tokenizer', 'config', 'history')):
                    try:
                        _sh.move(f'{OUT_DIR}/{_f}', f'{_arc}/{_f}')
                    except Exception:
                        pass
            RESUME_FROM = ''
            print(f'🔄 arsitektur baru {_new_arch} != checkpoint lama {_old_arch};')
            print(f'   checkpoint lama diarsipkan ke {_arc}; training dari NOL')
    except Exception as _e:
        print('⚠️ cek arsitektur checkpoint gagal:', _e)
if not RESUME_FROM:
    print('belum ada checkpoint - training dari awal')
print('device:', DEVICE)

## 4. Train
Kalau cell ini dijalankan ulang setelah sesi putus, ia **melanjutkan** dari
checkpoint (`model_latest.pt`) alih-alih mulai dari nol.

In [ ]:
!python -m aegisx.train \
    --data {DATA_DIR} \
    --out {OUT_DIR} \
    --vocab-size {VOCAB_SIZE} \
    --block-size {BLOCK_SIZE} \
    --n-layer {N_LAYER} \
    --n-head {N_HEAD} \
    --n-embd {N_EMBD} \
    --batch-size {BATCH_SIZE} \
    --grad-accum {GRAD_ACCUM} \
    --max-steps {MAX_STEPS} \
    --lr {LR} \
    --warmup-steps {WARMUP_STEPS} \
    --eval-every {EVAL_EVERY} \
    --save-every {SAVE_EVERY} \
    --early-stop-patience {EARLY_STOP} \
    --device {DEVICE} \
    {('--init-from ' + RESUME_FROM) if RESUME_FROM else ''}

## 5. Quick sanity check

In [ ]:
import os
if os.path.exists(f'{OUT_DIR}/model.pt'):
    !python -m aegisx.chat --model {OUT_DIR}/model.pt --tokenizer {OUT_DIR}/tokenizer.json \
        --prompt "You are AegisX, a cybersecurity assistant. User: how do I enumerate subdomains?\n\nAegisX:" \
        --max-new-tokens 150 --temperature 0.8 --top-k 50
else:
    print('Skipped: model.pt not found - check the training cell output for errors.')

## 5b. Evaluate (before you upload)

Runs the trained model against **20 fixed questions (10 EN + 10 ID)** and
prints a keyword-coverage score per question + per-language average.
Use this number to decide whether the model is worth uploading, and to
compare retrains (higher = better). A healthy *from-scratch* small model
will score modestly — that is expected and measurable.

In [ ]:
import os
if os.path.exists(f'{OUT_DIR}/model.pt'):
    !python -m aegisx.eval --model {OUT_DIR}/model.pt --tokenizer {OUT_DIR}/tokenizer.json \
        --device {DEVICE} --max-new-tokens 90
else:
    print('Skipped: model.pt not found - check the training cell output for errors.')

## 6. Package for manual Hugging Face upload

Copies `model.pt` + `tokenizer.json` + a ready model card + a **`knowledge/`**
folder (the corpus .txt files) into a clean export folder and creates a ZIP
you can download. **No auto-push** - you upload it yourself.

The `knowledge/` folder powers the **RAG grounding** feature in the Space app:
when present, answers are generated with retrieved references as context and
sources are shown under each reply.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

if not os.path.exists(f'{OUT_DIR}/model.pt'):
    print('Skipped: model.pt not found - nothing to export. Train first.')
else:
    EXPORT_DIR = Path('/content/drive/MyDrive/aegisx/export/aegisx-mini') if USE_DRIVE else Path('/content/aegisx/export/aegisx-mini')
    if EXPORT_DIR.exists():
        shutil.rmtree(EXPORT_DIR)
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    # 1. Copy model files
    shutil.copy(f'{OUT_DIR}/model.pt', EXPORT_DIR / 'model.pt')
    shutil.copy(f'{OUT_DIR}/tokenizer.json', EXPORT_DIR / 'tokenizer.json')
    shutil.copy(f'{OUT_DIR}/config.json', EXPORT_DIR / 'config.json')

    # 2. Model card (professional, lives in the repo at hf/MODEL_CARD.md)
    card_src = Path('hf/MODEL_CARD.md')
    if card_src.exists():
        shutil.copy(card_src, EXPORT_DIR / 'README.md')
        print('  model card copied from hf/MODEL_CARD.md')
    else:
        (EXPORT_DIR / 'README.md').write_text(
            '# AegisX-Mini\n\nA lightweight GPT-style model trained from scratch on public cybersecurity text.\n',
            encoding='utf-8',
        )

    # 3. Knowledge folder for RAG grounding in the Space app
    #    (only the raw .txt corpus; skips fetched JSON leftovers)
    KNOW_SRC = Path('data/raw')
    KNOW_DIR = EXPORT_DIR / 'knowledge'
    KNOW_DIR.mkdir(parents=True, exist_ok=True)
    n_know = 0
    for f in sorted(KNOW_SRC.glob('*.txt')):
        shutil.copy(f, KNOW_DIR / f.name)
        n_know += 1
    know_kb = sum(p.stat().st_size for p in KNOW_DIR.glob('*.txt'))
    print(f'  knowledge/ folder: {n_know} files, {know_kb/1024:.0f} KB (RAG grounding)')

    # 4. Zip it for easy download
    zip_path = Path(str(EXPORT_DIR) + '.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(EXPORT_DIR.iterdir()):
            if f.is_dir():
                for inner in sorted(f.rglob('*')):
                    if inner.is_file():
                        zf.write(inner, arcname=f'{f.name}/{inner.name}')
            else:
                zf.write(f, arcname=f.name)

    print('Export folder:')
    for f in sorted(EXPORT_DIR.iterdir()):
        if f.is_dir():
            print(f'  {f.name}/  ({sum(p.stat().st_size for p in f.rglob("*")):,} bytes)')
        else:
            print(f'  {f.name}  ({f.stat().st_size:,} bytes)')
    print(f'ZIP: {zip_path}')

## 7. Upload to Hugging Face (manual, 2 minutes)

**Download the export**: in the Files sidebar (left), browse to the export
folder (or the `.zip`) and download it. If you mounted Drive, it's at
`MyDrive/aegisx/export/aegisx-mini`.

**Option A — everything in one Space (simplest, enables RAG):**
1. In your existing Space (`aegisx-mini-space`), open the **Files** tab
2. Replace/add: `model.pt`, `tokenizer.json`, `config.json`
3. Upload `knowledge/` folder, `app.py` (= `hf/space_app.py`),
   `requirements.txt` (= `hf/requirements.txt`), and the `aegisx/` package
   folder (the app imports `aegisx.chat`, `aegisx.rag`, ...)
4. The Space rebuilds automatically. With `knowledge/` present, answers get
   grounded context + a **Sumber/Sources** line under each reply.

**Option B — clean split (model repo + Space loads it):**
1. https://huggingface.co/new → **Model** → name `aegisx-mini` → upload
   `model.pt`, `tokenizer.json`, `config.json`, `README.md`, and the
   `knowledge/` folder
2. In the Space: upload `app.py`, `requirements.txt`, `aegisx/` (no model
   files), then Settings → Variables and secrets → `AEGISX_REPO` =
   `FerzDevZ/aegisx-mini`

> Prefer the CLI later? `python3 hf/push_to_hub.py --repo FerzDevZ/aegisx-mini
> --checkpoint <export-folder>` (needs HF_TOKEN, optional).